# Monai Tryouts

First we import the different libraries

In [28]:
import os
import monai
import nibabel as nib  # Add nibabel import
from monai.data import Dataset, DataLoader, CacheDataset
from monai.transforms import (
    Compose, 
    LoadImaged, 
    EnsureChannelFirstd, 
    ScaleIntensityd, 
    ToTensord
)
import sys
sys.path.append('../../utils')
print(os.getcwd())
from utils import load_dataset

/Users/maxboc/Documents/Scia/S8/pinkcc/PinkCC-PinkPanthers/models/monai


Here we set the different directories paths

In [ ]:
data_dir = "../../data/DatasetChallenge"

# MSKCC
ct_mskcc_dir = os.path.join(data_dir, "CT", "MSKCC")
seg_mskcc_dir = os.path.join(data_dir, "Segmentation", "MSKCC")

# TCGA
ct_tcga_dir = os.path.join(data_dir, "CT", "TCGA")
seg_tcga_dir = os.path.join(data_dir, "Segmentation", "TCGA")

Function to find the associated CT-Scan with its segmentation

In [ ]:
# Helper function to get paired data with more verbose logging
def get_paired_data(ct_dir, seg_dir):
    if not os.path.exists(ct_dir):
        print(f"CT directory not found: {ct_dir}")
        return []
    
    if not os.path.exists(seg_dir):
        print(f"Segmentation directory not found: {seg_dir}")
        return []
    
    ct_files = [os.path.join(ct_dir, f) for f in os.listdir(ct_dir) if f.endswith('.nii.gz')]
    print(f"Found {len(ct_files)} CT files in {ct_dir}")
    
    paired_data = []
    
    for ct_file in ct_files:
        # Extract the filename without path and extension
        filename = os.path.basename(ct_file)
        base_filename = os.path.splitext(os.path.splitext(filename)[0])[0]  # Remove both .nii and .gz
        
        # Look for matching segmentation file
        matching_seg_files = [
            os.path.join(seg_dir, f)
            for f in os.listdir(seg_dir) 
            if f.startswith(base_filename) and f.endswith('.nii.gz')
        ]
        
        if matching_seg_files:
            # Use the first matching segmentation file
            paired_data.append({
                "image": ct_file,
                "label": matching_seg_files[0]
            })
            print(f"Paired: {ct_file} with {matching_seg_files[0]}")
        else:
            print(f"No matching segmentation found for {ct_file}")
    
    return paired_data

In [ ]:

# Get paired data from both sources
mskcc_data = get_paired_data(ct_mskcc_dir, seg_mskcc_dir)
tcga_data = get_paired_data(ct_tcga_dir, seg_tcga_dir)

In [ ]:

# Combine data from both sources
data = mskcc_data + tcga_data
print(f"Found {len(data)} paired CT and segmentation files")

if not data:
    print("No paired data found. Exiting.")
    import sys
    sys.exit(0)

# Define transforms for dictionary data
transform = Compose([
    LoadImaged(keys=["image", "label"]),  # Load both image and label
    EnsureChannelFirstd(keys=["image", "label"]),  # Ensure channel first for both
    ScaleIntensityd(keys=["image"]),  # Only normalize the image, not the label
    ToTensord(keys=["image", "label"])  # Convert both to tensors
])

# Create a regular Dataset first for debugging
print("Creating dataset...")
dataset = Dataset(data=data, transform=transform)

# Test loading the first item
try:
    print("Testing first item loading...")
    first_item = dataset[0]
    print(f"Successfully loaded first item. Image shape: {first_item['image'].shape}, Label shape: {first_item['label'].shape}")
except Exception as e:
    print(f"Error loading first item: {str(e)}")

# Create a DataLoader with only 1 worker for debugging
print("Creating dataloader...")
dataloader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)

# Example: Iterating through the dataset
try:
    print("Testing dataloader iteration...")
    for i, batch_data in enumerate(dataloader):
        images = batch_data["image"]
        labels = batch_data["label"]
        print(f"Batch {i}: Image shape: {images.shape}, Label shape: {labels.shape}")
        
        # Only process a few batches for testing
        if i >= 0:
            break
    print("Dataloader iteration successful")
except Exception as e:
    print(f"Error during dataloader iteration: {str(e)}")


In [12]:
import torchvision
import torch
from monai.networks.nets import UNet
from monai.networks.layers import Norm

# Define a 3D U-Net model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNet(
    spatial_dims=3,           # 3D data
    in_channels=1,          # Single channel input (CT)
    out_channels=2,         # Binary segmentation (background/foreground)
    channels=(16, 32, 64, 128, 256),  # Feature channels at each level
    strides=(2, 2, 2, 2),   # Stride for each down/upsampling
    norm=Norm.BATCH,        # Batch normalization
).to(device)

In [13]:
from monai.losses import DiceLoss

# Define loss function, optimizer, etc.
loss_function = DiceLoss(to_onehot_y=True, softmax=True)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
num_epochs = 100
val_interval = 5

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    step = 0
    
    for batch_data in train_loader:  # Create train and val loaders
        step += 1
        inputs, labels = batch_data["image"].to(device), batch_data["label"].to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        print(f"{step}/{len(train_loader)}, train_loss: {loss.item():.4f}")
    
    epoch_loss /= step
    print(f"epoch {epoch + 1} average loss: {epoch_loss:.4f}")
    
    # Validation
    if (epoch + 1) % val_interval == 0:
        model.eval()
        with torch.no_grad():
            for val_data in val_loader:
            val_inputs, val_labels = val_data["image"].to(device), val_data["label"].to(device)
            
            # Run sliding window inference for 3D volumes
            val_outputs = sliding_window_inference(
                inputs=val_inputs, 
                roi_size=roi_size, 
                sw_batch_size=sw_batch_size, 
                predictor=model,
                overlap=0.5
            )
            
            # Post-processing to convert to discrete segmentation masks
            val_outputs = [post_pred(i) for i in decollate_batch(val_outputs)]
            val_labels = [post_label(i) for i in decollate_batch(val_labels)]
            
            # Compute metrics
            dice_metric(y_pred=val_outputs, y=val_labels)
            hausdorff_metric(y_pred=val_outputs, y=val_labels)
            surface_distance_metric(y_pred=val_outputs, y=val_labels)
            
            progress_bar.update(1)
    progress_bar.close()


SyntaxError: incomplete input (2358049761.py, line 29)

In [ ]:
from monai.metrics import DiceMetric

# Set up validation metrics
dice_metric = DiceMetric(include_background=False, reduction="mean")

# In validation loop
val_outputs = model(val_inputs)
val_labels_list = [val_labels]
val_outputs_list = [val_outputs]
dice_metric(y_pred=val_outputs_list, y=val_labels_list)
mean_dice = dice_metric.aggregate().item()

In [ ]:
import matplotlib.pyplot as plt

# After training, visualize predictions on a validation sample
model.eval()
with torch.no_grad():
    val_data = val_dataset[0]
    val_inputs = val_data["image"].unsqueeze(0).to(device)
    val_labels = val_data["label"].unsqueeze(0).to(device)
    val_outputs = model(val_inputs)
    
    # Convert to numpy for visualization
    val_inputs = val_inputs[0, 0].cpu().numpy()
    val_labels = val_labels[0, 0].cpu().numpy()
    val_outputs = torch.argmax(val_outputs, dim=1)[0].cpu().numpy()
    
    # Plot a slice
    plt.figure(figsize=(15, 5))
    plt.subplot(1, 3, 1)
    plt.title("CT Image")
    plt.imshow(val_inputs[:, :, val_inputs.shape[2]//2], cmap="gray")
    plt.subplot(1, 3, 2)
    plt.title("Ground Truth")
    plt.imshow(val_labels[:, :, val_labels.shape[2]//2])
    plt.subplot(1, 3, 3)
    plt.title("Prediction")
    plt.imshow(val_outputs[:, :, val_outputs.shape[2]//2])
    plt.show()